#### NB4: RSE — Comparación Cerebro/Modelo

In [2]:
import numpy as np

valid_trs = np.load('valid_trs.npy')

In [3]:
PROMPTS = {
    'neutral': "",  # Sin instrucción, baseline
    'literary': (
        "You are a literary critic. As you read this text, focus on "
        "narrative structure, character development, and prose style."
    ),
    'cognitive': (
        "You are a cognitive scientist. As you read, focus on the mental "
        "states, beliefs, desires, and intentions of each character."
    ),
    'emotional': (
        "You are reading this story with deep emotional engagement. "
        "Focus on the emotions felt by each character and the emotional "
        "tension of each scene."
    ),
    'syntactic': (
        "You are a linguist analyzing sentence structure. Focus on "
        "grammatical complexity, dependency relations, and word order."
    ),
    'spatial': (
        'Pay close attention to physical movements, spatial locations, '
        'body positions, and the physical environment described in the text.'
    ),
}

**Generación de matrices RDM**

Se construyen las RDMs (Representational Dissimilarity Matrices) tanto para el cerebro como para cada condición del LLM, y se mide qué ondición de prompting produce representaciones más similares a las cerebrales.

Una RDM es una matriz cuadrada NxN donde N es el número de TRs válidos. La celda (i,j) mide la diferencia entre el patrón de actividad (cerebral o del modelo) entre el TR i y el TR j. Si dos condiciones (cerebro y modelo) generan RDMs parecidas, significa que organizan la información de forma similar: los momentos que el cerebro representa como "parecidos" son los mismos que el modelo representa como "parecidos".

Las RDMs cerebrales (una por sujeto) calculan la geometría representacional del cerebro. Se utilizan los patrones de actividad de los TRs válidos, calculados en el notebook anterior, y se calcula la distancia entre cada par de TRs (cada patrón de actividad corresponde a un vector de 10k vóxeles). Al ser matriz de disimilitud, se calcula mediante (1 - correlación de Pearson).

Se obtiene una matriz simétrica de dimensión (valid_trs, valid_trs). Si el valor es bajo, cercano a 0, los TRs tienen patrones similares. Por el contrario, si el valor es alto, significa que tienen patrones diferentes.

Las RDMs del modelo siguen la misma lógica, pero el patrón de actividad tiene dimensiones distintas. Se construye una por condición de prompt.

In [4]:
#RDMs cerebrales
from scipy.spatial.distance import pdist, squareform

brain_rdms = {}
for subj_id in range(1, 9):
    data = np.load(f'data_subject{subj_id}_isc10k.npy')
    brain_patterns = data[valid_trs]  # (n_valid_trs)

    # Distancia de correlación: 1 - r
    rdm = squareform(pdist(brain_patterns, metric='correlation'))
    brain_rdms[subj_id] = rdm
    print(f"Sujeto {subj_id}: RDM cerebral {rdm.shape}")

# RDM cerebral promedio (para análisis de grupo)
brain_rdm_mean = np.mean([brain_rdms[s] for s in brain_rdms], axis=0)

Sujeto 1: RDM cerebral (1283, 1283)
Sujeto 2: RDM cerebral (1283, 1283)
Sujeto 3: RDM cerebral (1283, 1283)
Sujeto 4: RDM cerebral (1283, 1283)
Sujeto 5: RDM cerebral (1283, 1283)
Sujeto 6: RDM cerebral (1283, 1283)
Sujeto 7: RDM cerebral (1283, 1283)
Sujeto 8: RDM cerebral (1283, 1283)


In [5]:
# Convierte valid_trs booleano a índices — añade esto UNA VEZ antes del bucle
valid_trs_idx = np.where(valid_trs)[0]

#RDMs cerebrales
from scipy.spatial.distance import pdist, squareform

brain_rdms = {}
for subj_id in range(1, 9):
    data = np.load(f'data_subject{subj_id}_isc10k.npy')
    brain_patterns = brain_patterns = data[valid_trs_idx]  # (n_valid_trs)

    # Distancia de correlación: 1 - r
    rdm = squareform(pdist(brain_patterns, metric='correlation'))
    brain_rdms[subj_id] = rdm
    print(f"Sujeto {subj_id}: RDM cerebral {rdm.shape}")

# RDM cerebral promedio (para análisis de grupo)
brain_rdm_mean = np.mean([brain_rdms[s] for s in brain_rdms], axis=0)

Sujeto 1: RDM cerebral (1283, 1283)
Sujeto 2: RDM cerebral (1283, 1283)
Sujeto 3: RDM cerebral (1283, 1283)
Sujeto 4: RDM cerebral (1283, 1283)
Sujeto 5: RDM cerebral (1283, 1283)
Sujeto 6: RDM cerebral (1283, 1283)
Sujeto 7: RDM cerebral (1283, 1283)
Sujeto 8: RDM cerebral (1283, 1283)


In [6]:
#RDMs de modelo
model_rdms = {}
for prompt_name in PROMPTS:
    tr_embeds = np.load(f'tr_embeddings_{prompt_name}.npy')
    model_patterns = tr_embeds[valid_trs]

    rdm = squareform(pdist(model_patterns, metric='correlation'))
    model_rdms[prompt_name] = rdm
    print(f"{prompt_name}: RDM modelo {rdm.shape}")

neutral: RDM modelo (1283, 1283)
literary: RDM modelo (1283, 1283)
cognitive: RDM modelo (1283, 1283)
emotional: RDM modelo (1283, 1283)
syntactic: RDM modelo (1283, 1283)
spatial: RDM modelo (1283, 1283)


In [7]:
model_rdms = {}
for prompt_name in PROMPTS:
    X_fir = np.load(f'fir_matrix_{prompt_name}.npy')   # ← cambia esta línea
    model_patterns = X_fir[valid_trs_idx]               # ← y esta

    rdm = squareform(pdist(model_patterns, metric='correlation'))
    model_rdms[prompt_name] = rdm
    print(f"{prompt_name}: RDM modelo {rdm.shape}")

neutral: RDM modelo (1283, 1283)
literary: RDM modelo (1283, 1283)
cognitive: RDM modelo (1283, 1283)
emotional: RDM modelo (1283, 1283)
syntactic: RDM modelo (1283, 1283)
spatial: RDM modelo (1283, 1283)


**RSA: Comparación cerebro/modelo**

Para cada condición de prompt, se calcula la correlación de Spearman entre las dos matrices RDM (sujeto y modelo). Cuanto más alta sea la correlación, más se parecen las representaciones entre ambos.

Si, por ejemplo, RSA(literary) > RSA(neutral), la condición de prompt "you are a literary critic" produce representaciones que se alinean mejor con el cerebro.

Se calcula por sujeto para poder hacer tests estadísticos.

In [8]:
from scipy.stats import spearmanr

results = []

for prompt_name in PROMPTS:
    model_rdm_vec = squareform(model_rdms[prompt_name])

    for subj_id in range(1, 9):
        brain_rdm_vec = squareform(brain_rdms[subj_id])

        rsa, pval = spearmanr(brain_rdm_vec, model_rdm_vec)
        results.append({
            'prompt': prompt_name,
            'subject': subj_id,
            'rsa': rsa,
            'pval': pval
        })

    # También con RDM promedio
    rsa_group, _ = spearmanr(squareform(brain_rdm_mean), model_rdm_vec)
    print(f"{prompt_name}: RSE grupo = {rsa_group:.4f}")

import pandas as pd
df_results = pd.DataFrame(results)
df_results.to_csv('rsa_results.csv', index=False)

neutral: RSE grupo = 0.0546
literary: RSE grupo = 0.0555
cognitive: RSE grupo = 0.0558
emotional: RSE grupo = 0.0556
syntactic: RSE grupo = 0.0547
spatial: RSE grupo = 0.0550


In [11]:
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist, squareform
import pandas as pd
import numpy as np

save_path = r'C:\Users\Ale\\'

df_results = pd.DataFrame(results)
df_results.to_csv(f'{save_path}rsa_results.csv', index=False)
print(f"\nGuardado: rsa_results.csv  ({len(df_results)} filas)")
print(df_results.groupby('prompt')['rsa'].mean().round(4))


Guardado: rse_results.csv  (48 filas)
prompt
cognitive    0.0232
emotional    0.0230
literary     0.0230
neutral      0.0225
spatial      0.0228
syntactic    0.0228
Name: rse, dtype: float64


### RSE GLOBAL: MÉTODO 2

In [1]:
import numpy as np, pandas as pd, scipy.io as sio
from scipy.stats import wilcoxon

save_path = 'C:/Users/Ale/'
PROMPTS = ['neutral','literary','cognitive','emotional','syntactic','spatial']

valid = np.load(save_path+'valid_trs.npy'); vidx = np.where(valid)[0]
runs  = sio.loadmat(save_path+'subject_1.mat', squeeze_me=True)['time'][:,1].astype(int)[vidx]

# Patrones cerebrales por sujeto (TRs válidos), z-score por vóxel en el tiempo
def zvox(Y):
    Y = Y[vidx]
    return (Y - Y.mean(0)) / (Y.std(0) + 1e-8)
brain = {s: zvox(np.load(save_path+f'data_subject{s}_isc10k.npy')) for s in range(1,9)}

# Precalcula, por condición, los pesos de similitud del modelo por fold (no dependen del sujeto)
def rse_folds(Xn, runs):
    F = []
    for te_run in np.unique(runs):
        te = runs == te_run; tr = ~te
        S = Xn[te] @ Xn[tr].T                 # similitud del modelo: test x train
        W = np.clip(S, 0, None)               # solo similitudes positivas
        W = W / (W.sum(1, keepdims=True) + 1e-12)
        F.append((te, tr, W))
    return F

# Síntesis del patrón cerebral y correlación predicho-real (RSE de Anderson)
def rse_score(F, Y):
    cs = []
    for te, tr, W in F:
        Yhat = W @ Y[tr]                       # patrón cerebral PREDICHO
        a = Yhat - Yhat.mean(1, keepdims=True)
        b = Y[te] - Y[te].mean(1, keepdims=True)
        cs.extend((a*b).sum(1) / (np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1) + 1e-12))
    return np.mean(cs)

rows = []
for p in PROMPTS:
    X  = np.load(save_path+f'fir_matrix_{p}.npy')[vidx]
    Xn = X - X.mean(1, keepdims=True); Xn /= (np.linalg.norm(Xn, axis=1, keepdims=True) + 1e-12)
    F  = rse_folds(Xn, runs)
    for s in range(1,9):
        rows.append({'prompt':p, 'subject':s, 'rse':rse_score(F, brain[s])})

df = pd.DataFrame(rows); df.to_csv(save_path+'rse_encoding_results.csv', index=False)
print(df.groupby('prompt')['rse'].mean().round(4))

print("\nWilcoxon vs neutral (¿el prompt mejora la predicción del cerebro?):")
for p in PROMPTS:
    if p == 'neutral': continue
    d = df[df.prompt==p].sort_values('subject').rse.values - df[df.prompt=='neutral'].sort_values('subject').rse.values
    stat, pv = wilcoxon(d, alternative='greater')
    print(f"  {p:10s}: ΔRSE = {d.mean():+.4f} ± {d.std():.4f}, p = {pv:.4f}")

prompt
cognitive   -0.0040
emotional   -0.0038
literary    -0.0053
neutral     -0.0038
spatial     -0.0040
syntactic   -0.0053
Name: rse, dtype: float64

Wilcoxon vs neutral (¿el prompt mejora la predicción del cerebro?):
  literary  : ΔRSE = -0.0015 ± 0.0005, p = 1.0000
  cognitive : ΔRSE = -0.0003 ± 0.0002, p = 0.9922
  emotional : ΔRSE = -0.0001 ± 0.0001, p = 0.8438
  syntactic : ΔRSE = -0.0015 ± 0.0005, p = 1.0000
  spatial   : ΔRSE = -0.0003 ± 0.0002, p = 1.0000


In [6]:
import numpy as np, pandas as pd, scipy.io as sio
from scipy.stats import wilcoxon

save_path = 'C:/Users/Ale/'
PROMPTS = ['neutral','literary','cognitive','emotional','syntactic','spatial']
ALPHA = 1e4   

valid = np.load(save_path+'valid_trs.npy'); vidx = np.where(valid)[0]
runs  = sio.loadmat(save_path+'subject_1.mat', squeeze_me=True)['time'][:,1].astype(int)[vidx]
def zvox(Y):
    Y = Y[vidx]; return (Y - Y.mean(0)) / (Y.std(0) + 1e-8)
brain = {s: zvox(np.load(save_path+f'data_subject{s}_isc10k.npy')) for s in range(1,9)}

def unwrap(x):
    while isinstance(x, np.ndarray) and x.size == 1: x = x.item()
    return x
roi_names    = np.array(unwrap(sio.loadmat(save_path+'subject_1.mat',squeeze_me=True)['meta']['ROInumToName']), dtype=object)
top_10k      = np.load(save_path+'isc_mask_indices.npy')
top_10k_rois = np.load(save_path+'voxel_rois.npy')[top_10k]
rois = {
 'Temporal_Mid_L':'MTG (semántica)', 'Angular_L':'Giro angular (integración)',
 'Frontal_Inf_Oper_L':'IFG Broca (sintaxis)', 'Temporal_Sup_L':'STG (lenguaje)',
 'Insula_L':'Ínsula izq. (emoción)', 'Insula_R':'Ínsula der. (emoción)',
 'Frontal_Inf_Orb_L':'OFC izq. (emoción/valor)',
 'Precuneus_L':'Precúneo izq. (DMN/espacial)', 'Precuneus_R':'Precúneo der. (DMN/espacial)',
 'Fusiform_R':'Fusiforme der. (forma visual)', 'Occipital_Mid_L':'Occipital med. (visual/espacial)',
 'Cingulum_Post_L':'PCC (red por defecto)',
}
roi_vox = {}
for r,d in rois.items():
    m = np.where(roi_names==r)[0]
    if len(m):
        v = np.where(top_10k_rois==m[0]+1)[0]
        if len(v) >= 20: roi_vox[d] = v

# Ridge en forma dual (más features que TRs): rápido y poca memoria
def build_ops(Xs, runs, alpha):
    ops = []
    for te_run in np.unique(runs):
        te = runs==te_run; tr = ~te
        Ktr = Xs[tr] @ Xs[tr].T
        Kte = Xs[te] @ Xs[tr].T
        M = Kte @ np.linalg.inv(Ktr + alpha*np.eye(Ktr.shape[0]))
        ops.append((te, tr, M))
    return ops
def voxel_acc(Y, ops):
    accs = []
    for te, tr, M in ops:
        Yp = M @ Y[tr]                                   # predicción del run dejado fuera
        a = Yp - Yp.mean(0); b = Y[te] - Y[te].mean(0)   # correlación por vóxel, sobre el tiempo
        accs.append((a*b).sum(0) / (np.linalg.norm(a,axis=0)*np.linalg.norm(b,axis=0) + 1e-12))
    return np.mean(accs, axis=0)                          # (V,) precisión por vóxel

glob, roirows = [], []
for p in PROMPTS:
    X  = np.load(save_path+f'fir_matrix_{p}.npy')[vidx]
    Xs = (X - X.mean(0)) / (X.std(0) + 1e-8)
    ops = build_ops(Xs, runs, ALPHA)
    for s in range(1,9):
        va = voxel_acc(brain[s], ops)
        glob.append({'prompt':p, 'subject':s, 'acc':va.mean()})
        for d, v in roi_vox.items():
            roirows.append({'roi':d, 'prompt':p, 'subject':s, 'acc':va[v].mean()})

dfg = pd.DataFrame(glob); dfr = pd.DataFrame(roirows)
dfg.to_csv(save_path+'encoding_global.csv', index=False); dfr.to_csv(save_path+'encoding_roi.csv', index=False)

print("Precisión de encoding global por prompt:")
print(dfg.groupby('prompt')['acc'].mean().round(4))
print("\nWilcoxon global vs neutral:")
for p in PROMPTS:
    if p=='neutral': continue
    d = dfg[dfg.prompt==p].sort_values('subject').acc.values - dfg[dfg.prompt=='neutral'].sort_values('subject').acc.values
    _, pv = wilcoxon(d, alternative='greater'); print(f"  {p:10s}: Δ={d.mean():+.4f}, p={pv:.4f}")

piv = dfr.groupby(['roi','prompt'])['acc'].mean().unstack()[PROMPTS]
print("\nPrecisión por ROI × prompt:\n", piv.round(4))
print("\nΔ vs neutral por ROI:\n", piv.sub(piv['neutral'], axis=0).drop(columns='neutral').round(4))

Precisión de encoding global por prompt:
prompt
cognitive    0.0247
emotional    0.0247
literary     0.0249
neutral      0.0243
spatial      0.0248
syntactic    0.0248
Name: acc, dtype: float64

Wilcoxon global vs neutral:
  literary  : Δ=+0.0006, p=0.0547
  cognitive : Δ=+0.0004, p=0.0977
  emotional : Δ=+0.0004, p=0.0742
  syntactic : Δ=+0.0005, p=0.1250
  spatial   : Δ=+0.0005, p=0.0547

Precisión por ROI × prompt:
 prompt                            neutral  literary  cognitive  emotional  \
roi                                                                         
Fusiforme der. (forma visual)      0.0168    0.0168     0.0166     0.0165   
Giro angular (integración)         0.0601    0.0614     0.0608     0.0612   
IFG Broca (sintaxis)               0.0301    0.0304     0.0300     0.0302   
MTG (semántica)                    0.0604    0.0616     0.0615     0.0615   
OFC izq. (emoción/valor)           0.0300    0.0299     0.0300     0.0301   
Occipital med. (visual/espacial)   0.0

In [5]:
import numpy as np, pandas as pd, scipy.io as sio
from scipy.stats import wilcoxon

save_path = 'C:/Users/Ale/'
PROMPTS = ['literary','cognitive','emotional','syntactic','spatial']   # cada uno vs neutral
ALPHA = 1e5

valid = np.load(save_path+'valid_trs.npy'); vidx = np.where(valid)[0]
runs  = sio.loadmat(save_path+'subject_1.mat', squeeze_me=True)['time'][:,1].astype(int)[vidx]
def zvox(Y):
    Y = Y[vidx]; return (Y - Y.mean(0)) / (Y.std(0) + 1e-8)
brain = {s: zvox(np.load(save_path+f'data_subject{s}_isc10k.npy')) for s in range(1,9)}

def zc(X): return (X - X.mean(0)) / (X.std(0) + 1e-8)
neu_raw = np.load(save_path+'fir_matrix_neutral.npy')[vidx]
Xc = zc(neu_raw)                       # espacio CONTENIDO (lectura base)
folds = [(runs==r, runs!=r) for r in np.unique(runs)]

def r2_voxel(Y, grams, alpha):
    out = []
    for (te, tr, Ktr, Kte) in grams:
        M = Kte @ np.linalg.inv(Ktr + alpha*np.eye(Ktr.shape[0]))
        Yp = M @ Y[tr]; yte = Y[te]
        res = ((yte - Yp)**2).sum(0); tot = ((yte - yte.mean(0))**2).sum(0) + 1e-12
        out.append(1 - res/tot)
    return np.mean(out, axis=0)

gC = [(te, tr, Xc[tr]@Xc[tr].T, Xc[te]@Xc[tr].T) for te, tr in folds]   # grams contenido (una vez)

rows = []
for p in PROMPTS:
    Xp = zc(np.load(save_path+f'fir_matrix_{p}.npy')[vidx] - neu_raw)   # espacio ESPECÍFICO del prompt
    gJ = [(te, tr, KcTr + Xp[tr]@Xp[tr].T, KcTe + Xp[te]@Xp[tr].T) for (te, tr, KcTr, KcTe) in gC]
    for s in range(1,9):
        r2c = r2_voxel(brain[s], gC, ALPHA).mean()
        r2j = r2_voxel(brain[s], gJ, ALPHA).mean()
        rows.append({'prompt':p, 'subject':s, 'r2_contenido':r2c, 'r2_conjunto':r2j, 'unica':r2j - r2c})

df = pd.DataFrame(rows)
print("Varianza única del prompt (ΔR² = conjunto − contenido):")
print(df.groupby('prompt')[['r2_contenido','r2_conjunto','unica']].mean().round(5))
print("\n¿La varianza única es > 0 entre sujetos? (Wilcoxon + FDR)")
pv = []
for p in PROMPTS:
    _, pp = wilcoxon(df[df.prompt==p]['unica'].values, alternative='greater'); pv.append(pp)
pv = np.array(pv); n = len(pv); o = np.argsort(pv); q = np.empty(n)
q[o] = np.clip(np.minimum.accumulate((pv[o]*n/(np.arange(n)+1))[::-1])[::-1], 0, 1)
for p, pp, qq in zip(PROMPTS, pv, q):
    flag = "  <-- varianza única significativa" if qq < 0.05 else ""
    print(f"  {p:10s}: ΔR²={df[df.prompt==p]['unica'].mean():+.5f}, p={pp:.4f}, q={qq:.4f}{flag}")

Varianza única del prompt (ΔR² = conjunto − contenido):
           r2_contenido  r2_conjunto    unica
prompt                                       
cognitive      -0.00424     -0.00837 -0.00412
emotional      -0.00424     -0.00774 -0.00349
literary       -0.00424     -0.00893 -0.00469
spatial        -0.00424     -0.00836 -0.00412
syntactic      -0.00424     -0.00933 -0.00509

¿La varianza única es > 0 entre sujetos? (Wilcoxon + FDR)
  literary  : ΔR²=-0.00469, p=1.0000, q=1.0000
  cognitive : ΔR²=-0.00412, p=1.0000, q=1.0000
  emotional : ΔR²=-0.00349, p=1.0000, q=1.0000
  syntactic : ΔR²=-0.00509, p=1.0000, q=1.0000
  spatial   : ΔR²=-0.00412, p=1.0000, q=1.0000
